In [1]:
import os

In [2]:
%pwd

'/Users/srishtisingh/chest-cancer-classification-mlops/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/srishtisingh/chest-cancer-classification-mlops'

In [5]:

import os


os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/SrishtiSingh100/chest-cancer-classification-mlops.mlflow/#/"
os.environ["MLFLOW_TRACKING_USERNAME"] = "SrishtiSingh100"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "5c15a38015192da271ee09c636aca5caa73d0da9"

print("✓ MLflow environment variables set")

✓ MLflow environment variables set


In [6]:
import tensorflow as tf

In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

2026-01-12 21:36:30.255147: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-01-12 21:36:30.255170: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-12 21:36:30.255174: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-12 21:36:30.255225: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-12 21:36:30.255454: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/Chest-CT-Scan-data",
            mlflow_uri= "https://dagshub.com/SrishtiSingh100/chest-cancer-classification-mlops.mlflow/#/",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config
    

In [11]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

/opt/anaconda3/envs/chestmlops/lib/python3.10/site-packages/mlflow/utils/requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [12]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")

In [13]:
# Copy the corrected code to your file
import shutil

# This will update your model_evaluation_mlflow.py file
# Just copy the entire content from the artifact above and save it to:
# src/cnnClassifier/components/model_evaluation_mlflow.py

print("Please copy the code from the artifact above to your model_evaluation_mlflow.py file")

Please copy the code from the artifact above to your model_evaluation_mlflow.py file


In [14]:
# Fresh imports after kernel restart
from cnnClassifier.config.configuration import ConfigurationManager
from cnnClassifier.components.model_evaluation_mlflow import Evaluation

try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
    
    print("✓ Success! Check ./mlruns folder")

except Exception as e:
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()

[2026-01-12 21:36:30,934: INFO: common: yaml file: /Users/srishtisingh/chest-cancer-classification-mlops/config/config.yaml loaded successfully]
[2026-01-12 21:36:30,935: INFO: common: yaml file: /Users/srishtisingh/chest-cancer-classification-mlops/params.yaml loaded successfully]
[2026-01-12 21:36:30,935: INFO: common: created directory at: artifacts]
[2026-01-12 21:36:30,935: INFO: common: created directory at: artifacts/evaluation]
Found 102 images belonging to 2 classes.


2026-01-12 21:36:31.233307: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


7/7 [==============================] - 1s 147ms/step - loss: 26.9843 - accuracy: 0.5686
[2026-01-12 21:36:32,447: INFO: common: json file saved at: scores.json]


2026/01/12 21:36:34 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /var/folders/v6/rj0rjt8x0c9_f_dcfqgnsg6r0000gn/T/tmputlnl_rv/model/data/model/assets
[2026-01-12 21:36:34,951: INFO: builder_impl: Assets written to: /var/folders/v6/rj0rjt8x0c9_f_dcfqgnsg6r0000gn/T/tmputlnl_rv/model/data/model/assets]
Error: module 'tensorflow.keras' has no attribute '__version__'


Traceback (most recent call last):
  File "/var/folders/v6/rj0rjt8x0c9_f_dcfqgnsg6r0000gn/T/ipykernel_39709/2620302867.py", line 10, in <module>
    evaluation.log_into_mlflow()
  File "/Users/srishtisingh/chest-cancer-classification-mlops/src/cnnClassifier/components/model_evaluation_mlflow.py", line 107, in log_into_mlflow
    mlflow.keras.log_model(
  File "/opt/anaconda3/envs/chestmlops/lib/python3.10/site-packages/mlflow/tensorflow/__init__.py", line 222, in log_model
    return Model.log(
  File "/opt/anaconda3/envs/chestmlops/lib/python3.10/site-packages/mlflow/models/model.py", line 551, in log
    flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)
  File "/opt/anaconda3/envs/chestmlops/lib/python3.10/site-packages/mlflow/tensorflow/__init__.py", line 445, in save_model
    "keras_version": keras_module.__version__,
AttributeError: module 'tensorflow.keras' has no attribute '__version__'


In [15]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
    
    print("✓ Success! Check ./mlruns folder")

except Exception as e:
    raise e

[2026-01-12 21:36:35,461: INFO: common: yaml file: /Users/srishtisingh/chest-cancer-classification-mlops/config/config.yaml loaded successfully]
[2026-01-12 21:36:35,462: INFO: common: yaml file: /Users/srishtisingh/chest-cancer-classification-mlops/params.yaml loaded successfully]
[2026-01-12 21:36:35,463: INFO: common: created directory at: artifacts]
[2026-01-12 21:36:35,464: INFO: common: created directory at: artifacts/evaluation]
Found 102 images belonging to 2 classes.


2026-01-12 21:36:35.780560: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


7/7 [==============================] - 1s 139ms/step - loss: 26.9843 - accuracy: 0.5686
[2026-01-12 21:36:36,928: INFO: common: json file saved at: scores.json]


2026/01/12 21:36:38 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /var/folders/v6/rj0rjt8x0c9_f_dcfqgnsg6r0000gn/T/tmpevnk9w1r/model/data/model/assets
[2026-01-12 21:36:38,872: INFO: builder_impl: Assets written to: /var/folders/v6/rj0rjt8x0c9_f_dcfqgnsg6r0000gn/T/tmpevnk9w1r/model/data/model/assets]


2026/01/12 21:36:43 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /var/folders/v6/rj0rjt8x0c9_f_dcfqgnsg6r0000gn/T/tmpevnk9w1r/model, flavor: tensorflow), fall back to return ['tensorflow==2.13.1']. Set logging level to DEBUG to see the full traceback.
/opt/anaconda3/envs/chestmlops/lib/python3.10/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Successfully registered model 'VGG16Model'.
2026/01/12 21:37:09 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.


✓ Success! Check ./mlruns folder
